# Lire un fichier Parquet

Environnement Conda `mlops`. Les fichiers téléchargés vont dans `data/` et ne sont pas versionnés.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-01.parquet"
file_path = data_dir / "yellow_tripdata_2021-01.parquet"

if not file_path.exists():
    urlretrieve(url, file_path)

file_path

In [ ]:
import pandas as pd

df = pd.read_parquet(file_path)
df.head()

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df[["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID"]].head()

## Plusieurs mois

Télécharge janvier et février 2021, puis rassemble tous les fichiers `yellow_tripdata_*.parquet` présents dans `data/`.

In [ ]:
urls = {
    "2021-01": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-01.parquet",
    "2021-02": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-02.parquet",
}

for month, url in urls.items():
    destination = data_dir / f"yellow_tripdata_{month}.parquet"
    if not destination.exists():
        urlretrieve(url, destination)

In [ ]:
files = sorted(data_dir.glob("yellow_tripdata_*.parquet"))
frames = []

for parquet_file in files:
    month_frame = pd.read_parquet(parquet_file)
    source_month = parquet_file.stem.removeprefix("yellow_tripdata_")
    month_frame["source_month"] = source_month
    frames.append(month_frame)

trips = pd.concat(frames, ignore_index=True)
trips.head()

In [ ]:
trips.groupby("source_month").size()

In [ ]:
counts = trips.groupby("source_month").size()
counts.sum() == len(trips)